# 03 · Le format Excel

Le service financier veut les courses de mars « dans un fichier Excel ». Excel est parfait pour qu'un humain lise un tableau, mais il a des limites : découvrons-les avec pandas.

## 1. Charger les courses

On part du fichier Parquet du mois, téléchargé au notebook 00.

In [1]:
import zipfile
from pathlib import Path

import pandas as pd

DATA = Path("../data")
OUTPUT = Path("../output")
OUTPUT.mkdir(exist_ok=True)

courses = pd.read_parquet(DATA / "yellow_tripdata_2026-03.parquet")
courses.shape

(3952451, 20)

Près de 4 millions de courses (3 952 451) et 20 colonnes.

## 2. Tout le mois dans Excel ?

On essaie d'écrire tout le mois dans un fichier Excel. Le `try/except` affiche l'erreur sans arrêter le notebook.

In [2]:
try:
    courses.to_excel(OUTPUT / "courses_2026-03.xlsx", index=False)
except ValueError as erreur:
    print(erreur)

This sheet is too large! Your sheet size is: 3952451, 20 Max sheet size is: 1048576, 16384


Une feuille Excel s'arrête à **1 048 576 lignes** : notre mois en a presque 4 fois plus. pandas refuse tout de suite, sans rien écrire.

## 3. Écrire un échantillon

On garde les 50 000 premières courses. Excel s'écrit lentement : patientez. `%%time` mesure la durée de la cellule (*Wall time* = durée réelle écoulée).

In [3]:
%%time
chemin_xlsx = OUTPUT / "courses_echantillon.xlsx"
echantillon = courses.head(50_000)
echantillon.to_excel(chemin_xlsx, index=False)
print(round(chemin_xlsx.stat().st_size / 1_000_000, 1), "Mo")

4.3 Mo
CPU times: user 15.8 s, sys: 228 ms, total: 16 s
Wall time: 16 s


Plus de 10 secondes, et 4,3 Mo pour seulement 50 000 lignes. Le Parquet du mois complet, avec 80 fois plus de lignes, ne pèse que 68 Mo.

## 4. Un `.xlsx` est un zip de XML

Un fichier Excel est en fait une archive zip remplie de fichiers XML (du texte avec des balises). On affiche ses 2 premiers octets, puis la liste des fichiers de l'archive.

In [4]:
print(chemin_xlsx.read_bytes()[:2])
zipfile.ZipFile(chemin_xlsx).namelist()

b'PK'


['docProps/app.xml',
 'docProps/core.xml',
 'xl/theme/theme1.xml',
 'xl/worksheets/sheet1.xml',
 'xl/styles.xml',
 '_rels/.rels',
 'xl/workbook.xml',
 'xl/_rels/workbook.xml.rels',
 '[Content_Types].xml']

`PK` est la signature de tout fichier zip. Nos données sont dans `xl/worksheets/sheet1.xml` ; les autres fichiers décrivent le classeur et ses styles.

## 5. Relire : deux moteurs

Par défaut, pandas lit Excel avec le moteur *openpyxl*. Le moteur *calamine* fait le même travail : comparons.

In [5]:
%%time
relu = pd.read_excel(chemin_xlsx)

CPU times: user 7.94 s, sys: 25.3 ms, total: 7.97 s
Wall time: 7.97 s


In [6]:
%%time
relu = pd.read_excel(chemin_xlsx, engine="calamine")

CPU times: user 1.18 s, sys: 40 ms, total: 1.22 s
Wall time: 1.22 s


Calamine relit le même fichier plusieurs fois plus vite. Comparons maintenant les types avant et après le passage par Excel.

In [7]:
pd.DataFrame({"Parquet": echantillon.dtypes, "Excel relu": relu.dtypes}).head(4)

,Parquet,Excel relu
VendorID,int32,int64
tpep_pickup_datetime,datetime64[us],datetime64[us]
tpep_dropoff_datetime,datetime64[us],datetime64[us]
passenger_count,float64,int64


Les dates reviennent en dates : contrairement au CSV, Excel garde un type pour chaque cellule. Mais il n'a qu'un seul type « nombre » : `VendorID` et `passenger_count` changent de type.

## 6. Plusieurs feuilles dans un classeur

La finance veut une feuille par *borough* (arrondissement de New York). On ajoute d'abord le *borough* de départ grâce à la table des zones.

In [8]:
zones = pd.read_csv(DATA / "taxi_zone_lookup.csv")
echantillon = echantillon.merge(zones, left_on="PULocationID", right_on="LocationID")
echantillon["Borough"].value_counts()

Borough
Manhattan        43571
Queens            4828
Brooklyn          1139
Bronx              316
Unknown             73
EWR                 12
Staten Island        5
Name: count, dtype: int64

Près de 9 courses sur 10 partent de Manhattan (43 571 sur 50 000).

`ExcelWriter` ouvre un seul classeur, et chaque `to_excel` y ajoute une feuille. Un nom de feuille ne peut pas dépasser 31 caractères, d'où `borough[:31]`. Comptez une vingtaine de secondes ou plus : la cellule n'affiche rien.

In [9]:
with pd.ExcelWriter(OUTPUT / "courses_par_borough.xlsx") as writer:
    for borough, d in echantillon.groupby("Borough"):
        d.to_excel(writer, sheet_name=borough[:31], index=False)

Pour relire, `sheet_name=None` charge toutes les feuilles d'un coup et renvoie un dictionnaire `{nom de la feuille: DataFrame}`.

In [10]:
feuilles = pd.read_excel(OUTPUT / "courses_par_borough.xlsx", sheet_name=None, engine="calamine")
for nom, feuille in feuilles.items():
    print(nom, ":", len(feuille), "lignes")

Bronx : 316 lignes
Brooklyn : 1139 lignes
EWR : 12 lignes
Manhattan : 43571 lignes
Queens : 4828 lignes
Staten Island : 5 lignes
Unknown : 73 lignes


7 feuilles, une par *borough*, avec les mêmes nombres de courses qu'au-dessus.

## Exercices

### ✍️ Exercice 1 — Un petit rapport

Créez `output/exo03_rapport.xlsx` avec deux feuilles : « Par borough » (nombre de courses et moyenne de `total_amount`) et « Par paiement » (même chose par `payment_type`). Relisez-le avec `sheet_name=None`.

In [11]:
# ✅ Solution
par_borough = echantillon.groupby("Borough")["total_amount"].agg(nb_courses="count", montant_moyen="mean").round(2)
par_paiement = echantillon.groupby("payment_type")["total_amount"].agg(nb_courses="count", montant_moyen="mean").round(2)

with pd.ExcelWriter(OUTPUT / "exo03_rapport.xlsx") as writer:
    par_borough.to_excel(writer, sheet_name="Par borough")
    par_paiement.to_excel(writer, sheet_name="Par paiement")

rapport = pd.read_excel(OUTPUT / "exo03_rapport.xlsx", sheet_name=None)
print(list(rapport))
rapport["Par paiement"]

['Par borough', 'Par paiement']


,payment_type,nb_courses,montant_moyen
0,1,43078,28.70
1,2,5968,24.08
2,3,262,10.11
3,4,692,2.52


Le classeur relu contient bien nos 2 feuilles. Dans l'échantillon, la carte (`payment_type` 1) domine : 43 078 courses sur 50 000, pour 28,70 $ en moyenne.

### ✍️ Exercice 2 — Lire seulement un coin du fichier

Relisez `courses_echantillon.xlsx` en ne gardant que les colonnes A à E et les 5 premières lignes (paramètres `usecols` et `nrows`).

In [12]:
# ✅ Solution
pd.read_excel(chemin_xlsx, usecols="A:E", nrows=5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance
0,2,2026-03-01 00:02:26,2026-03-01 00:13:45,1,2.58
1,2,2026-03-01 00:19:33,2026-03-01 00:28:21,1,1.50
2,2,2026-03-01 00:07:20,2026-03-01 00:15:12,2,0.88
3,2,2026-03-01 00:16:11,2026-03-01 00:28:20,1,1.76
4,2,2026-03-01 00:20:47,2026-03-01 00:30:44,2,1.57


`usecols` accepte les lettres de colonnes d'Excel. Avec `nrows`, pandas s'arrête après 5 lignes : la réponse est immédiate, alors que relire tout le fichier prenait plusieurs secondes.

## À retenir

- Une feuille Excel contient au plus 1 048 576 lignes : on y met des échantillons ou des résumés, pas des données brutes.
- Un `.xlsx` est un zip de fichiers XML : lent à écrire et à relire ; `engine="calamine"` relit plusieurs fois plus vite.
- Excel garde les dates, mais n'a qu'un seul type « nombre ».
- `ExcelWriter` écrit plusieurs feuilles ; `sheet_name=None` les relit toutes.
- **Excel en entrée et en sortie, pour les humains ; jamais au milieu d'un pipeline.**